# 4. Reflection

*Using Microsoft Semantic Kernel (Agent Framework)*

Implements a reflection pattern where agents can review and critique their own outputs. This self-improvement mechanism helps generate higher-quality responses through iterative refinement.

In [ ]:
import os
from dotenv import load_dotenv
import semantic_kernel as sk
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.contents import ChatHistory

load_dotenv()

kernel = sk.Kernel()
service_id = "chat-gpt"
kernel.add_service(
    AzureChatCompletion(
        service_id=service_id,
        deployment_name="gpt-4o-mini",
        endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    )
)

chat_service = kernel.get_service(service_id)

In [ ]:
async def generate_with_reflection(prompt: str, max_iterations: int = 2) -> str:
    """Generate content with self-reflection and improvement."""
    
    # Initial generation
    chat_history = ChatHistory()
    chat_history.add_system_message("You are a helpful assistant that generates high-quality content.")
    chat_history.add_user_message(prompt)
    
    response = await chat_service.get_chat_message_content(
        chat_history=chat_history,
        settings=kernel.get_prompt_execution_settings_from_service_id(service_id),
    )
    
    current_output = str(response)
    print(f"Initial Output:\n{current_output}\n")
    
    # Reflection loop
    for i in range(max_iterations):
        # Reflect on the output
        reflection_history = ChatHistory()
        reflection_history.add_system_message(
            "You are a critic that provides constructive feedback to improve content."
        )
        reflection_history.add_user_message(
            f"Review this response and suggest improvements:\n\n{current_output}"
        )
        
        reflection = await chat_service.get_chat_message_content(
            chat_history=reflection_history,
            settings=kernel.get_prompt_execution_settings_from_service_id(service_id),
        )
        
        print(f"Reflection {i+1}:\n{reflection}\n")
        
        # Improve based on reflection
        improvement_history = ChatHistory()
        improvement_history.add_system_message("You are a helpful assistant that improves content based on feedback.")
        improvement_history.add_user_message(
            f"Original prompt: {prompt}\n\nCurrent response: {current_output}\n\nFeedback: {reflection}\n\nProvide an improved response:"
        )
        
        improved = await chat_service.get_chat_message_content(
            chat_history=improvement_history,
            settings=kernel.get_prompt_execution_settings_from_service_id(service_id),
        )
        
        current_output = str(improved)
        print(f"Improved Output {i+1}:\n{current_output}\n")
    
    return current_output

In [ ]:
# Test reflection
final_output = await generate_with_reflection(
    "Write a short paragraph about the benefits of AI agents."
)

print("\n=== FINAL OUTPUT ===")
print(final_output)